In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import os

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets

from src.dataset import get_transforms, make_loaders
from src.evaluate import get_logits, predict_loader, summarize
from src.experiment import run_experiment, run_experiment_tl
from src.model import CNNBaseline, TLResNet, describe_run
from src.train import fit, make_scheduler
from src.utils import get_device, set_seed

SEED, EPOCHS, IMG_SIZE, BATCH = 42, 30, 128, 32
DATA = '../data/processed'
RESULTS = '../results/ablations.csv'
device = get_device()
paths = {
    'data': '../data/processed',
    'ckpt_dir': '../checkpoints',
    'log_dir': '../logs',
    'results': '../results/experiments.csv',
}

In [ ]:
img_size = 224
_, eval_tf = get_transforms(img_size, augment=False)
train_ds = datasets.ImageFolder('../data/processed/train', transform=eval_tf)
val_ds   = datasets.ImageFolder('../data/processed/val', transform=eval_tf)
g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=g)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

In [ ]:
set_seed(SEED)

model = TLResNet(trainable=('fc',)).to(device)
opt = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

res_fe = run_experiment_tl(
    name='resnet_fe',
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=opt,
    criterion=nn.CrossEntropyLoss(),
    device=device,
    epochs=20,
    extra={'lr_head': 1e-3, 'trainable': 'fc', 'img_size': 224},
)

In [12]:
set_seed(SEED)

model = TLResNet(trainable=('fc',)).to(device)
ckpt = torch.load('../checkpoints/resnet_fe.pt', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])   # ← اینجا
model.set_trainable(('layer4', 'fc'))
opt = torch.optim.Adam([
    {'params': model.net.layer4.parameters(), 'lr': 1e-4},
    {'params': model.net.fc.parameters(),     'lr': 1e-3},
])

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,}")

res_ft = run_experiment_tl(
    name='resnet_ft',
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=opt,
    criterion=nn.CrossEntropyLoss(),
    device=device,
    epochs=20,
    extra={'lr_layer4': 1e-4, 'lr_head': 1e-3, 'trainable': 'layer4+fc', 'img_size': 224},
)

print(res_ft)

trainable: 8,397,832 / 11,180,616
{'epoch': 1, 'train_loss': 0.29347605365860596, 'val_loss': 0.23987519776285682, 'val_macro_f1': 0.9040287957588246, 'lr': 0.0001}
{'epoch': 2, 'train_loss': 0.026288662195126287, 'val_loss': 0.21110282722975063, 'val_macro_f1': 0.9113237795355627, 'lr': 0.0001}
{'epoch': 3, 'train_loss': 0.0069061714233339825, 'val_loss': 0.2075250243512698, 'val_macro_f1': 0.9115415148533142, 'lr': 0.0001}
{'epoch': 4, 'train_loss': 0.0035205542112939398, 'val_loss': 0.20997628794837717, 'val_macro_f1': 0.9151172391766369, 'lr': 0.0001}
{'epoch': 5, 'train_loss': 0.0018346312208665791, 'val_loss': 0.21037969526660852, 'val_macro_f1': 0.9134025055559796, 'lr': 0.0001}
{'epoch': 6, 'train_loss': 0.0020240923131872885, 'val_loss': 0.20562746771446144, 'val_macro_f1': 0.9172591626462483, 'lr': 0.0001}
{'epoch': 7, 'train_loss': 0.002091384347519397, 'val_loss': 0.21761638988205728, 'val_macro_f1': 0.91949618672749, 'lr': 0.0001}
{'epoch': 8, 'train_loss': 0.0013601436118